# 02 · 神经网络模块与训练循环 nn.Module & Training Loop

> **能力标签**：SH8（PyTorch 基础 / PyTorch fundamentals）

## 目标 Objectives

完成本课后，你应该能够：

1. 继承 `nn.Module` 构建自定义神经网络，理解 `__init__` 与 `forward` 的职责分工。
2. 使用 `nn.Sequential`、`nn.Linear`、`nn.ReLU` 快速组合前馈网络。
3. 选择优化器（`torch.optim.SGD` / `Adam`）与损失函数（`nn.MSELoss` / `nn.CrossEntropyLoss`）。
4. 手写完整训练循环：`zero_grad` → 正向传播 → 计算损失 → `backward` → `step`。
5. 理解 `optimizer.zero_grad()` 为何必须在每步迭代前调用。

> 对应能力：**SH8**


## 概念讲解 Concepts

### 1. nn.Module：网络的基类 Base Class

所有 PyTorch 神经网络都继承自 `nn.Module`：

```python
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()           # 必须调用
        self.fc = nn.Linear(2, 1)   # 注册子模块（参数自动追踪）

    def forward(self, x):
        return self.fc(x)            # 定义正向计算
```

调用 `net(x)` 会自动调用 `net.forward(x)`，同时触发钩子（hook）等机制。

---

### 2. nn.Sequential：快速组合 Quick Composition

```python
net = nn.Sequential(
    nn.Linear(in_dim, hidden),
    nn.ReLU(),
    nn.Linear(hidden, out_dim),
)
```

等价于手写 `forward`，但更简洁。层按顺序串联，输入从第一层流向最后一层。

---

### 3. 参数访问 Parameter Access

| 方法 | 说明 |
|------|------|
| `model.parameters()` | 生成器，返回所有可训练参数（`Tensor`） |
| `model.named_parameters()` | 返回 `(name, param)` 对 |
| `model.state_dict()` | 所有参数的字典（用于保存/加载） |

---

### 4. 训练循环 Training Loop

完整步骤（与 nanograd n3 一一对应）：

```python
for step in range(steps):
    optimizer.zero_grad()     # 1. 清零梯度
    pred = model(x)           # 2. 正向传播
    loss = loss_fn(pred, y)   # 3. 计算损失
    loss.backward()           # 4. 反向传播
    optimizer.step()          # 5. 更新参数
```

---

### 5. 为何必须 zero_grad()？

PyTorch 梯度**累加**（与 nanograd 的 `grad += ...` 相同）。
若不清零，历史梯度会与当前步骤的梯度叠加，导致更新方向错误。

| nanograd n3                          | PyTorch                     |
|--------------------------------------|-----------------------------|
| `for p in params: p.grad = 0.0`     | `optimizer.zero_grad()`     |
| `loss.backward()`                   | `loss.backward()`           |
| `for p in params: p.data -= lr*p.grad` | `optimizer.step()`       |


## 示例 Worked Example — MLP + train_linear

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ── 镜像 w6-nn-module 练习：MLP 类 ───────────────────────────────────────
class MLP(nn.Module):
    # 两层 MLP：Linear(in,hidden)->ReLU->Linear(hidden,out)
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_out),
        )

    def forward(self, x):
        return self.net(x)


# ── 验证模型结构 ──────────────────────────────────────────────────────────
model = MLP(4, 8, 2)
print("MLP(4, 8, 2) 结构:")
print(model)
print()

n_params = sum(p.numel() for p in model.parameters())
# Linear(4,8): 32 w + 8 b = 40; Linear(8,2): 16 w + 2 b = 18; total = 58
expected = 4*8 + 8 + 8*2 + 2
print(f"参数总数: {n_params}  (期望 {expected})")
assert n_params == expected, f"参数数错误: {n_params}"
print("✓ 模型结构正确")
print()

x = torch.randn(3, 4)
y = model(x)
print(f"输入形状: {x.shape}  ->  输出形状: {y.shape}  (应为 [3, 2])")
assert y.shape == (3, 2)
print("✓ 正向传播形状正确")


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ── 镜像 w6-training-loop 练习：train_linear 函数 ─────────────────────────
def train_linear(steps=200, lr=0.1, seed=0):
    # 在 y=2x+1 的合成数据上训练 nn.Linear(1,1)，返回 (model, final_loss)
    torch.manual_seed(seed)
    x = torch.linspace(-1, 1, 64).unsqueeze(1)
    y = 2 * x + 1

    model = nn.Linear(1, 1)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loss = None
    for step in range(steps):
        opt.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        opt.step()
    return model, float(loss)


# 训练并验证
model, final_loss = train_linear(steps=200, lr=0.1, seed=0)
w = model.weight.item()
b = model.bias.item()

print(f"训练 200 步后：")
print(f"  最终损失   final_loss = {final_loss:.6f}  (期望 < 0.01)")
print(f"  学到的权重 weight    = {w:.4f}  (目标 ≈ 2.0)")
print(f"  学到的偏置 bias      = {b:.4f}  (目标 ≈ 1.0)")
assert final_loss < 0.01,     f"损失太高: {final_loss}"
assert abs(w - 2.0) < 0.05,  f"权重偏差过大: {w}"
assert abs(b - 1.0) < 0.05,  f"偏置偏差过大: {b}"
print("✓ 线性回归训练成功")


In [ ]:
# ── 可视化损失曲线 Loss Curve ─────────────────────────────────────────────
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os

torch.manual_seed(0)
x_data = torch.linspace(-1, 1, 64).unsqueeze(1)
y_data = 2 * x_data + 1

model_vis = nn.Linear(1, 1)
opt_vis   = torch.optim.SGD(model_vis.parameters(), lr=0.1)
loss_fn   = nn.MSELoss()

losses = []
for _ in range(200):
    opt_vis.zero_grad()
    loss = loss_fn(model_vis(x_data), y_data)
    loss.backward()
    opt_vis.step()
    losses.append(loss.item())

tmpdir = tempfile.mkdtemp()
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses)
ax.set_xlabel("训练步 Step")
ax.set_ylabel("MSE Loss")
ax.set_title("训练损失曲线 Training Loss Curve (y=2x+1)")
ax.set_yscale("log")
fig.tight_layout()
fig.savefig(os.path.join(tmpdir, "loss_curve.png"), dpi=80)
plt.close(fig)

print(f"损失曲线已保存至: {os.path.join(tmpdir, 'loss_curve.png')}")
print(f"初始损失: {losses[0]:.4f}  ->  最终损失: {losses[-1]:.6f}")
print("损失曲线单调下降，证明训练循环正确。")


## 动手 Exercise

In [ ]:
# ── 动手练习：用 Adam 优化器训练 MLP 做二分类 ──────────────────────────────
import torch
import torch.nn as nn

torch.manual_seed(42)

N = 200
x_train = torch.randn(N, 2)
y_train  = (x_train[:, 0] - x_train[:, 1] > 0).float().unsqueeze(1)


class BinaryMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)


for opt_name in ["SGD", "Adam"]:
    torch.manual_seed(42)
    model = BinaryMLP()
    if opt_name == "SGD":
        opt = torch.optim.SGD(model.parameters(), lr=0.1)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=0.01)

    loss_fn = nn.BCELoss()
    for step in range(200):
        opt.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        loss.backward()
        opt.step()

    with torch.no_grad():
        pred = (model(x_train) > 0.5).float()
        acc = (pred == y_train).float().mean().item()

    print(f"{opt_name:4s}  最终损失={loss.item():.4f}  训练准确率={acc:.3f}")

print()
print("观察：Adam 通常比 SGD 收敛更快（对学习率不那么敏感）。")


## 延伸阅读 Further Reading

1. **PyTorch nn.Module 文档**：<https://pytorch.org/docs/stable/generated/torch.nn.Module.html>
2. **PyTorch 优化器指南**：<https://pytorch.org/docs/stable/optim.html>
3. **Sebastian Ruder «Gradient Descent Optimization Algorithms»**：<https://ruder.io/optimizing-gradient-descent/> — SGD、Momentum、Adam 的系统综述。
4. **《Deep Learning with PyTorch》** Stevens et al. 第 5-6 章 — nn.Module 与训练循环的详细讲解。


## 关联练习 Related Assignments

```bash
python check.py w6-nn-module
python check.py w6-training-loop
```

> **w6-nn-module**：实现 `MLP(n_in, n_hidden, n_out)` 类，`forward` 调用 `nn.Sequential`。

> **w6-training-loop**：实现 `train_linear(steps, lr, seed)` 在 y=2x+1 数据上训练 `nn.Linear(1,1)` 并返回 `(model, final_loss)`。

> 能力标签：**SH8** · threshold ≥ 0.7
